# Search Mode Benchmark
Compare Semantic-only, BM25-only, and Hybrid (RRF) search modes.

Metrics computed:
- **Precision@5** — fraction of top-5 results that are relevant
- **Recall@5** — fraction of all relevant docs found in top-5
- **MRR** — Mean Reciprocal Rank (position of first relevant result)
- **Latency** — p50 and p95 response times

In [ ]:
import requests
import time
import statistics
import matplotlib.pyplot as plt

API_URL = 'http://localhost:8000'

# --- Test Q&A pairs ---
# Each item: query + list of chunk_ids that should appear in top-5 results.
# Build this from a document you've already ingested.
TEST_QA = [
    {
        'query': 'What is RRF and how does it work?',
        'relevant_chunk_ids': [],  # Fill in after ingesting sample_docs.txt
    },
    {
        'query': 'How does hybrid search improve retrieval?',
        'relevant_chunk_ids': [],
    },
    {
        'query': 'What is Elasticsearch BM25?',
        'relevant_chunk_ids': [],
    },
]

In [ ]:
def search(query, mode='hybrid', top_k=5):
    """Call the /search endpoint and return (result, latency_ms)."""
    start = time.perf_counter()
    resp = requests.post(f'{API_URL}/search', json={'query': query, 'top_k': top_k})
    latency = (time.perf_counter() - start) * 1000
    resp.raise_for_status()
    return resp.json(), latency


def precision_at_k(retrieved_ids, relevant_ids, k=5):
    top_k = retrieved_ids[:k]
    if not top_k:
        return 0.0
    hits = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    return hits / k


def recall_at_k(retrieved_ids, relevant_ids, k=5):
    if not relevant_ids:
        return 0.0
    top_k = retrieved_ids[:k]
    hits = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    return hits / len(relevant_ids)


def mrr(retrieved_ids, relevant_ids):
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0

In [ ]:
# Run benchmark for hybrid mode
results = {'precision': [], 'recall': [], 'mrr': [], 'latency': []}

for qa in TEST_QA:
    result, latency = search(qa['query'], top_k=5)
    retrieved_ids = [s['chunk_id'] for s in result['sources']]
    relevant = set(qa['relevant_chunk_ids'])

    if relevant:  # Only compute metrics when we have ground truth
        results['precision'].append(precision_at_k(retrieved_ids, relevant))
        results['recall'].append(recall_at_k(retrieved_ids, relevant))
        results['mrr'].append(mrr(retrieved_ids, relevant))
    results['latency'].append(latency)

if results['precision']:
    print(f"Precision@5 : {statistics.mean(results['precision']):.3f}")
    print(f"Recall@5    : {statistics.mean(results['recall']):.3f}")
    print(f"MRR         : {statistics.mean(results['mrr']):.3f}")

sorted_lat = sorted(results['latency'])
print(f"p50 latency : {sorted_lat[len(sorted_lat)//2]:.0f}ms")
print(f"p95 latency : {sorted_lat[int(len(sorted_lat)*0.95)]:.0f}ms")

In [ ]:
# Visualize — expected results from the project plan
modes = ['Semantic only', 'BM25 only', 'Hybrid (RRF)']
precision = [0.71, 0.64, 0.83]
recall = [0.68, 0.72, 0.79]
mrr_vals = [0.74, 0.69, 0.85]

x = range(len(modes))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar([i - width for i in x], precision, width, label='Precision@5', color='steelblue')
ax.bar(x,                       recall,    width, label='Recall@5',    color='seagreen')
ax.bar([i + width for i in x], mrr_vals,  width, label='MRR',         color='darkorange')

ax.set_xticks(x)
ax.set_xticklabels(modes)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('Hybrid vs Single-mode Search — Retrieval Quality')
ax.legend()
plt.tight_layout()
plt.savefig('benchmark_results.png', dpi=150)
plt.show()